# 対戦の記録・再生（リプレイ）機能一式を扱う

対戦を最後まで進めて `Battle.build_replay_data()` で対戦の記録を取り、`replay_battle()`
でその記録から同じ展開の対戦をそのまま再生する（`ReplayPlayer` 等の詳細は
docs/api/README.md の リプレイ系 節を参照）。「興味深い対戦を記録して後で解析する」
という戦術研究ユースケースの入口。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

[▶ Google Colabで開く](https://colab.research.google.com/github/tmwork1/jpoke/blob/main/examples/05_advanced/02_replay.ipynb)

In [ ]:
%pip install -q jpoke

In [ ]:
from jpoke import Battle, Player
from jpoke.core.replay import replay_battle

In [ ]:
player1 = Player("Player 1")
player1.add_pokemon("ピカチュウ", moves=["かみなり"])

player2 = Player("Player 2")
player2.add_pokemon("フシギダネ", moves=["たいあたり"])

battle = Battle(player1, player2, seed=1)
# 手動でstep()するループの定型は01で学んだので、ここではplay_out()で開始から一括して進める
battle.play_out(max_turns=100)
winner = battle.winner
print(f"元の対戦の勝者: {winner.username if winner else '引き分け（ターン上限）'}（{battle.turn}ターン）")

In [ ]:
# build_replay_data()は対戦の途中でも呼べるが、ここでは決着後に呼ぶ
# （詳細は docs/api/README.md の リプレイ系 節を参照）
replay_data = battle.build_replay_data()

# to_dict()/from_dict() で辞書化・復元できる。json.dumps()/json.loads() を
# 挟めばファイル保存や別プロセスへの受け渡しもできる
serialized = replay_data.to_dict()
restored = type(replay_data).from_dict(serialized)

In [ ]:
# replay_battle()の挙動（ReplayPlayerによる自動進行・乱数シードの扱い）は
# docs/api/README.md の リプレイ系 節を参照
replayed_battle = replay_battle(restored)

In [ ]:
# 勝者・ターン数の一致だけでなく、get_log_lines(turn="all")（詳細は
# docs/api/README.md の Battle「ログ系」節を参照）が完全に一致することを確認し、
# 「同じ展開が再現された」ことを実際に検証する
original_log_lines = battle.get_log_lines(turn="all")
replayed_log_lines = replayed_battle.get_log_lines(turn="all")
assert original_log_lines == replayed_log_lines, "リプレイの再生結果が元の対戦と一致しない"

replayed_winner = replayed_battle.winner
print(
    f"再生した対戦の勝者: "
    f"{replayed_winner.username if replayed_winner else '引き分け（ターン上限）'}"
    f"（{replayed_battle.turn}ターン）"
)
print(
    f"ログ行数: 元={len(original_log_lines)}行 / 再生={len(replayed_log_lines)}行 "
    "（完全一致することをassertで確認済み）"
)

試してみよう: serialized を json.dumps() でファイルに保存し、別プロセスで
json.load() → from_dict() → replay_battle() すると、対戦の記録・共有・
後日の再解析ができる